In [1]:
# =============================================================================
# STOCK MARKET SIGNAL: PREDICT NEXT DAY RETURNS
# Kaggle Competition Solution
# =============================================================================
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/stock-market-signal-predict-next-day-returns/sample_submission.csv
/kaggle/input/competitions/stock-market-signal-predict-next-day-returns/train.csv
/kaggle/input/competitions/stock-market-signal-predict-next-day-returns/test.csv


In [2]:

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import polars as pl
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')


In [3]:
# ── 1. VERİYİ OKUMA ──────────────────────────────────────────────────────────
df_train = pd.read_csv('/kaggle/input/competitions/stock-market-signal-predict-next-day-returns/train.csv')
df_test  = pd.read_csv('/kaggle/input/competitions/stock-market-signal-predict-next-day-returns/test.csv')


In [4]:
# ── 2. VERİNİN İLK BAKIŞI ────────────────────────────────────────────────────
print("--- Train: İlk 5 Satır ---")
print(df_train.head())

print("\n--- Test: İlk 5 Satır ---")
print(df_test.head())

print("\n--- Sütun Bilgileri ve Eksik Değerler ---")
print(df_train.info())

print("\n--- İstatistiksel Özet ---")
print(df_train.describe())


--- Train: İlk 5 Satır ---
   id   stock_id  return_1d  return_5d  return_10d  return_20d  sma_ratio_5  \
0   0  stock_053  -0.037127  -0.007780    0.068146    0.075390     0.979739   
1   1  stock_055   0.000106  -0.000627    0.018637    0.032935     1.009760   
2   2  stock_041   0.021354   0.001978   -0.039911   -0.006946     0.999311   
3   3  stock_028   0.025042   0.045190    0.054391    0.078166     1.023532   
4   4  stock_043   0.012005   0.053654    0.046700   -0.291112     1.002994   

   sma_ratio_10  sma_ratio_20  sma_ratio_50  ...  volume_sma_ratio_20  \
0      1.015518      1.041614      1.099135  ...             0.950214   
1      1.006498      1.003074      1.050365  ...             1.186582   
2      0.986230      0.971957      0.979119  ...             1.095516   
3      1.038882      1.053964      1.045069  ...             0.899269   
4      1.021734      0.993001      0.807702  ...             0.361751   

   daily_range  avg_range_10d  high_low_ratio  momentum_10d

In [5]:
# ── 3. ÖZELLİK MÜHENDİSLİĞİ ─────────────────────────────────────────────────
def optimize_features(df):
    """
    Finansal sinyaller üzerinde Polars ile hızlı özellik mühendisliği:
      - Volatilite × Momentum etkileşimi
      - RSI × Bollinger Band pozisyonu (trend gücü sinyali)
      - Hacim ivmesi (kısa/uzun vadeli hacim oranları arası fark)
      - Kısa vadeli ortalama getiri
    """
    df_pl = pl.from_pandas(df)

    df_pl = df_pl.with_columns([
        # Volatilite ve Momentum Etkileşimi
        (pl.col('volatility_20d') * pl.col('momentum_20d')).alias('vol_mom_interaction'),

        # Göreceli Güç × Bollinger Konumu  →  trendin hem gücü hem konumu
        (pl.col('rsi_14') * pl.col('bb_position')).alias('rsi_bb_signal'),

        # Hacim İvmesi  →  kısa vadeli hacim artışı uzun vadeye göre
        (pl.col('volume_sma_ratio_10') / (pl.col('volume_sma_ratio_20') + 1e-6)).alias('volume_acceleration'),

        # Kısa Vadeli Ortalama Getiri  →  1 günlük + 5 günlük ortalaması
        ((pl.col('return_1d') + pl.col('return_5d')) / 2).alias('short_term_avg_return'),
    ])

    return df_pl.to_pandas()


# Hedef ve özellik ayrımı
drop_cols = ['id', 'stock_id', 'target']
features  = [c for c in df_train.columns if c not in drop_cols]

X      = optimize_features(df_train)
y      = df_train['target']
X_test = optimize_features(df_test)


In [6]:
#─ 4. MODELLEME: 5-FOLD XGBOOST ─────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=False)

params = {
    'objective'           : 'binary:logistic',
    'eval_metric'         : 'auc',
    'max_depth'           : 8,       # Gürültülü finansal veride aşırı derinliğe gitme
    'learning_rate'       : 0.01,
    'subsample'           : 0.8,
    'colsample_bytree'    : 0.8,
    'min_child_weight'    : 5,       # Overfitting önleme
    'reg_lambda'          : 10,      # L2 Regülarizasyonu
    'early_stopping_rounds': 100,
    'random_state'        : 42,
    'tree_method'         : 'hist',  # GPU/CPU için hızlı histogram yöntemi
    'n_jobs'              : -1,
}

print(f"Eğitim başlıyor... Toplam Örnek: {len(X)}")

oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[train_idx][features], X.iloc[val_idx][features]
    y_tr, y_va = y.iloc[train_idx],           y.iloc[val_idx]

    model = xgb.XGBClassifier(**params, n_estimators=1000)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=100,
    )

    fold_preds          = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx]  = fold_preds
    test_preds         += model.predict_proba(X_test[features])[:, 1] / kf.get_n_splits()

    print(f"Fold {fold + 1} AUC: {roc_auc_score(y_va, fold_preds):.5f}")

Eğitim başlıyor... Toplam Örnek: 440402
[0]	validation_0-auc:0.52614
[100]	validation_0-auc:0.53971
[200]	validation_0-auc:0.54096
[300]	validation_0-auc:0.54222
[400]	validation_0-auc:0.54363
[500]	validation_0-auc:0.54433
[600]	validation_0-auc:0.54497
[700]	validation_0-auc:0.54531
[800]	validation_0-auc:0.54563
[900]	validation_0-auc:0.54612
[999]	validation_0-auc:0.54627
Fold 1 AUC: 0.54638
[0]	validation_0-auc:0.52425
[100]	validation_0-auc:0.54067
[200]	validation_0-auc:0.54101
[300]	validation_0-auc:0.54190
[400]	validation_0-auc:0.54265
[500]	validation_0-auc:0.54319
[600]	validation_0-auc:0.54356
[700]	validation_0-auc:0.54396
[800]	validation_0-auc:0.54428
[900]	validation_0-auc:0.54435
[999]	validation_0-auc:0.54427
Fold 2 AUC: 0.54442
[0]	validation_0-auc:0.52517
[100]	validation_0-auc:0.53731
[200]	validation_0-auc:0.53845
[300]	validation_0-auc:0.53943
[400]	validation_0-auc:0.54016
[500]	validation_0-auc:0.54078
[600]	validation_0-auc:0.54141
[700]	validation_0-auc:0.54

In [7]:

# ── 5. SONUÇLAR VE SUBMISSION ─────────────────────────────────────────────────
total_auc = roc_auc_score(y, oof_preds)
print(f"\nGenel OOF AUC: {total_auc:.5f}")

submission = pd.DataFrame({'id': df_test['id'], 'target': test_preds})
submission.to_csv('submission.csv', index=False)
print("submission.csv kaydedildi ✓")


Genel OOF AUC: 0.54329
submission.csv kaydedildi ✓
